# Silver Layer — Data Cleaning & Quality Validation

**Databricks Pattern:** Silver = clean, conformed, validated data.

This notebook demonstrates:
- Type coercion and null handling with PySpark
- Flagging suspicious records without dropping them
- Schema validation with Pandera
- Quality reporting


In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from src.utils.spark_session import get_spark_session
from src.utils.paths import BRONZE_PATH, SILVER_PATH
from pyspark.sql import functions as F

spark = get_spark_session()

In [ ]:
# ── Run Silver pipeline ───────────────────────────────────────────────────────
from src.pipelines.silver import main as run_silver
run_silver()

In [ ]:
# ── Compare Bronze vs Silver: what changed? ───────────────────────────────────
bronze = spark.read.format('delta').load(str(BRONZE_PATH / 'orders'))
silver = spark.read.format('delta').load(str(SILVER_PATH / 'orders'))

print(f'Bronze rows: {bronze.count():,}')
print(f'Silver rows: {silver.count():,}')
print(f'Suspicious flagged: {silver.filter(F.col("_is_negative_amount") | F.col("_is_future_order") | F.col("_is_extreme_amount")).count():,}')

In [ ]:
# ── Show flagged suspicious orders ───────────────────────────────────────────
silver.filter(
    F.col('_is_negative_amount') | F.col('_is_future_order') | F.col('_is_extreme_amount')
).select(
    'order_id', 'total_amount', 'order_timestamp',
    '_is_negative_amount', '_is_future_order', '_is_extreme_amount'
).show(10, truncate=False)

In [ ]:
# ── Read quality report ───────────────────────────────────────────────────────
import json
from pathlib import Path
from src.utils.paths import REPORTS_PATH

reports = sorted(REPORTS_PATH.glob('silver_quality_*.json'))
if reports:
    with open(reports[-1]) as f:
        report = json.load(f)
    print(f'Overall quality score: {report["overall_quality_score"]}%')
    for table, stats in report['table_stats'].items():
        print(f"  {table}: {stats['rows_before']:,} → {stats['rows_after']:,} rows")

In [ ]:
# ── Null analysis ─────────────────────────────────────────────────────────────
from pyspark.sql.functions import col, count, when, isnan

null_counts = silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ['customer_id', 'order_timestamp', 'total_amount', 'status', 'payment_method']
])
null_counts.show()